# Trigger wave thresholds

In [2]:
import immunowave as iw
import numpy as np
import jax
import jax.numpy as jnp
import diffrax as dx

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from multiprocessing import Pool
import multiprocessing as mp
from functools import partial

import pickle

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")

from hill_function_utils import (
    tissue_response,
    single_cell_response,
    compute_wave_threshold,
    compute_cell_threshold,
)

import warnings

warnings.simplefilter("ignore")

from immunowave_paper_utils import style_axes, colors, fontsize, linewidth, rc_params
from diffrax import SaveAt

In [3]:
%matplotlib qt

In [4]:
linewidth = 3
fontsize = 24
markersize = 12
markeredgewidth = 2
rc_params["axes.linewidth"] = linewidth
rc_params["font.size"] = fontsize
mpl.rcParams.update(rc_params)
mpl.rcParams["pdf.fonttype"] = 42
sim_color = np.array([94, 45, 144]) / 255

In [10]:
"""ok, so multiprocessing with jax only works if you use a
non-default start method within the multiprocessing library.
Here I use forkserver. An annoying corollary of forkserver
(and also spawn) is that in jupyter notebooks, functions
called within a Pool() need to be imported and can't be
defined in another cell. So I moved the functions to a
python file in this same directory."""

mp.set_start_method("forkserver")

## Model definition
Here the model is defined with slightly different notation from the main text:

$\partial_t A = D\partial^2_x A + \frac{A^n}{K_D^n + A^n} - \gamma A + \eta B\delta(x)$

In this notebook we solve the corresponding ODE
$\frac{dA}{dt} = \frac{A^n}{K_D^n + A^n} - \gamma A + \eta B$

by using the same PDE solver but on a grid of size 1 and no diffusion.

In [7]:
def Bc_cell_theory(n, KD, gamma):
    return (1 - 1 / n) * (1 / n) ** (1 / (n - 1)) * (KD * gamma) ** (n / (n - 1))


In [11]:
class State4(iw.State):
    A: iw.ScalarField
    B: iw.ScalarField


class HillModel(iw.Model):
    KD: float
    n: int
    η: float
    ξ: float
    λ: float
    μ: float
    D: float = 1.0
    gamma: float = 1.0

    @jax.jit
    def __call__(self, t, state, args=None):
        # unpack field variables
        A, B = state.A, state.B
        # unpack parameters
        KD, n, η, ξ, λ, μ, D, gamma = (
            self.KD,
            self.n,
            self.η,
            self.ξ,
            self.λ,
            self.μ,
            self.D,
            self.gamma,
        )
        # define PDE
        An = A.binop(n, jnp.power)
        tmp = An + KD**n
        tmp = tmp.binop(-1, jnp.power)
        hill_term = An * tmp
        dAdt = D * A.laplacian(bc="neumann") + hill_term - gamma * A + η * B
        dBdt = ξ * B.laplacian(bc="neumann") + λ * B * (1 - B) - μ * A * B
        return State4(dAdt, dBdt)


# @jax.jit
def response(B0, KD, t_max, hill_coefficient=2):
    model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0)
    state = State4(
        A=iw.ScalarField(shape, lb, h, 0),
        B=iw.ScalarField(
            shape,
            lb,
            h,
            fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=1),
        ),
    )
    solution = iw.solve(
        model,
        state,
        t0=0,
        t1=t_max,
        t=jnp.array([t_max]),
        **kwargs,
    )
    return np.sum(solution.ys.A.values[-1] * solution.ys.A.h)
    # return solution.evaluate(t_final).A.integral() / L


def find_B0(final_mean_A, B0s):
    this_id = np.where(np.diff(final_mean_A) == np.max(np.diff(final_mean_A)))[0][0]
    B0c = 0.5 * (B0s[this_id] + B0s[this_id + 1])
    uncertainty = 0.5 * (B0s[this_id + 1] - B0s[this_id])
    return B0c, uncertainty

### Vary $K_D$

In [9]:
KDs = np.logspace(-2, -0.5, 20)
B0s = np.logspace(-4, 0, 40)
t_max = 100
L = 100.0
n = 2000  # 200
D = 1.0
gamma = 1.0
hill_coefficient = 4  # [2, 3, 4, 5]
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5

"""mixed"""
print("mixed")
# run the model on a grid of length 1 with no diffusion
L = 1.0
n = 2
shape = (n,)
lb = [0]
h = L / (n - 1)

Bc_cell = np.zeros(len(KDs))
Bc_cell_uncertainties = np.zeros(len(KDs))
for i, KD in enumerate(KDs):
    print(f"{i + 1} of {len(KDs)}")
    Bc_cell[i], Bc_cell_uncertainties[i] = compute_cell_threshold(
        B0s,
        KD=KDs[i],
        t_max=t_max,
        hill_coefficient=hill_coefficient,
        D=D,
        gamma=gamma,
        shape=shape,
        lb=lb,
        h=h,
        L=L,
        n_iters=n_iters,
        **kwargs,
    )

single cell
1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

0.0008376776400682916


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

0.0010150459007658775


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

0.10311377481214981


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

0.10311391591825218


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [12]:
KD_sweep_dict = {
    "KDs": KDs,
    "Bc_cell": Bc_cell,
    "Bc_cell_uncertainties": Bc_cell_uncertainties,
}
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/KD_sweep_cell.pkl",
    "wb",
) as f:
    pickle.dump(KD_sweep_dict, f)

In [8]:
"""plot"""

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/KD_sweep_cell.pkl",
    "rb",
) as f:
    KD_sweep_dict = pickle.load(f)
KDs = KD_sweep_dict["KDs"]
Bc_cell = KD_sweep_dict["Bc_cell"]
hill_coefficient = 4

plt.figure(figsize=(8, 7))

plt.errorbar(
    KDs,
    Bc_cell,
    marker="d",
    markersize=18,
    markerfacecolor="none",
    markeredgewidth=4,
    linestyle="none",
    markeredgecolor=colors["cell"],
    label="cell numerics",
)

cell_theory = (
    (1 - (1 / hill_coefficient))
    * (1 / hill_coefficient) ** (1 / (hill_coefficient - 1))
    * KDs ** (hill_coefficient / (hill_coefficient - 1))
)
plt.plot(KDs, cell_theory, "-", linewidth=4, color=colors["cell"], label="cell theory")

plt.legend(fontsize=18)
plt.xscale("log")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel(
    "concentration scale \nof positive feedback, $K_D$ (a.u.)", fontsize=fontsize
)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()

### Vary hill coefficients

In [15]:
"""vary hill coefficients"""

KD = 0.01  
B0s = np.logspace(-5, -2, 40)
t_max = 100
L = 100.0
n = 2000
hill_coefficients = np.linspace(2, 5, 20) 
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5


"""single cell"""
print("single cell")
# run the model on a grid of length 1 with no diffusion
L = 1.0
n = 2
shape = (n,)
lb = [0]
h = L / (n - 1)

Bc_cell = np.zeros(len(hill_coefficients))
Bc_cell_uncertainties = np.zeros(len(hill_coefficients))
for i in range(len(hill_coefficients)):
    print(f"{i + 1} of {len(hill_coefficients)}")
    Bc_cell[i], Bc_cell_uncertainties[i] = compute_cell_threshold(
        B0s,
        KD=KD,
        t_max=t_max,
        hill_coefficient=hill_coefficients[i],
        D=D,
        gamma=gamma,
        shape=shape,
        lb=lb,
        h=h,
        L=L,
        n_iters=n_iters,
        **kwargs,
    )

single cell
1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

2.4244620170823306e-05


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

2.508785831273124e-05


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

0.001693009628046046


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

0.001693009628046046


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [16]:
hill_coeff_sweep_dict = {
    "hill_coefficients": hill_coefficients,
    "Bc_cell": Bc_cell,
    "Bc_cell_uncertainties": Bc_cell_uncertainties,
}
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/hill_coefficient_sweep.pkl",
    "wb",
) as f:
    pickle.dump(hill_coeff_sweep_dict, f)

In [9]:
"""plot"""

ns = np.linspace(2, 5, 100)

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/hill_coefficient_sweep.pkl",
    "rb",
) as f:
    hill_coeff_sweep_dict = pickle.load(f)
hill_coefficients = hill_coeff_sweep_dict["hill_coefficients"]
Bc_cell = hill_coeff_sweep_dict["Bc_cell"]

plt.figure(figsize=(8, 7))
plt.errorbar(
    hill_coefficients,
    Bc_cell,
    marker="d",
    markersize=18,
    markerfacecolor="none",
    markeredgewidth=4,
    linestyle="none",
    markeredgecolor=colors["cell"],
    label="cell numerics",
)

cell_theory = (1 - (1 / ns)) * (1 / ns) ** (1 / (ns - 1)) * KD ** (ns / (ns - 1))
plt.plot(ns, cell_theory, "-", linewidth=4, color=colors["cell"], label="cell theory")

plt.legend(fontsize=18)
plt.xscale("linear")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel("hill coefficient, $n$", fontsize=fontsize)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()

### Vary decay rate

In [89]:
"""vary decay rate. now time in minutes. length in microns"""

B0s = np.logspace(-5, -4, 40)
KD = 0.01  
t_maxs = [1000]  
L = 100.0
n = 2000
hill_coefficient = 3
gammas = [0.2]
D = 1.0
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5

"""single cell"""
print("single cell")
# run the model on a grid of length 1 with no diffusion
L = 1.0
n = 2
shape = (n,)
lb = [0]
h = L / (n - 1)

Bc_cell = np.zeros(len(gammas))
Bc_cell_uncertainties = np.zeros(len(gammas))
for i in range(len(gammas)):
    print(f"{i + 1} of {len(gammas)}")
    Bc_cell[i], Bc_cell_uncertainties[i] = compute_cell_threshold(
        B0s,
        KD=KD,
        t_max=t_maxs[i],
        hill_coefficient=hill_coefficient,
        D=D,
        gamma=gammas[i],
        shape=shape,
        lb=lb,
        h=h,
        L=L,
        n_iters=n_iters,
        **kwargs,
    )

single cell
1 of 1


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3.257020655659783e-05


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3.439869860828185e-05


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3.443940523058813e-05


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3.443940523058813e-05


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [90]:
"""vary decay rate. now time in minutes. length in microns"""

B0s = np.logspace(-6, -3, 40)
KD = 0.01  # np.logspace(-2, -0.5, 5)
t_maxs = np.linspace(20000, 1000, 20)
L = 100.0
n = 2000
hill_coefficient = 3
gammas = np.linspace(0.02, 0.2, 20)
D = 1.0
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5

"""single cell"""
print("single cell")
# run the model on a grid of length 1 with no diffusion
L = 1.0
n = 2
shape = (n,)
lb = [0]
h = L / (n - 1)

Bc_cell = np.zeros(len(gammas))
Bc_cell_uncertainties = np.zeros(len(gammas))
for i in range(len(gammas)):
    print(f"{i + 1} of {len(gammas)}")
    Bc_cell[i], Bc_cell_uncertainties[i] = compute_cell_threshold(
        B0s,
        KD=KD,
        t_max=t_maxs[i],
        hill_coefficient=hill_coefficient,
        D=D,
        gamma=gammas[i],
        shape=shape,
        lb=lb,
        h=h,
        L=L,
        n_iters=n_iters,
        **kwargs,
    )

single cell
1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

1e-06


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

1.0844667412601385e-06


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3.443941337283044e-05


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3.443941337283044e-05


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [91]:
gamma_sweep_dict = {
    "gammas": gammas,
    "Bc_cell": Bc_cell,
    "Bc_cell_uncertainties": Bc_cell_uncertainties,
}
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/gamma_sweep.pkl",
    "wb",
) as f:
    pickle.dump(gamma_sweep_dict, f)

In [12]:
"""plot"""
# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/gamma_sweep.pkl",
    "rb",
) as f:
    gamma_sweep_dict = pickle.load(f)
gammas = gamma_sweep_dict["gammas"]
Bc_cell = gamma_sweep_dict["Bc_cell"]
gamma_theory = np.linspace(np.min(gammas), np.max(gammas), 1000)
hill_coefficient = 3

plt.figure(figsize=(8, 7))
plt.errorbar(
    gammas,
    Bc_cell,
    marker="d",
    markersize=18,
    markerfacecolor="none",
    markeredgewidth=4,
    linestyle="none",
    markeredgecolor=colors["cell"],
    label="cell numerics",
)

cell_theory = (
    (1 - (1 / hill_coefficient))
    * (1 / hill_coefficient) ** (1 / (hill_coefficient - 1))
    * (KD * gamma_theory) ** (hill_coefficient / (hill_coefficient - 1))
)
plt.plot(
    gamma_theory,
    cell_theory,
    "-",
    linewidth=4,
    color=colors["cell"],
    label="cell theory",
)

plt.legend(fontsize=18)
plt.xscale("linear")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel(r"loss rate, $\gamma$ (1/min)", fontsize=24)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()

## Plot everything including schematics

In [23]:
f, axs = plt.subplots(2, 3, figsize=(18.97, 10.79))

### Solution branches schematic

In [24]:
A = np.linspace(0, 1, 10000)
KD = 0.4
hill_coefficient = 4
B = Bc_cell_theory(hill_coefficient, KD, 1)
f1 = A**hill_coefficient / (KD**hill_coefficient + A**hill_coefficient)
f2 = A**hill_coefficient / (KD**hill_coefficient + A**hill_coefficient) + B
f3 = A

ax = axs[0, 0]
ax.plot(A, f1, linewidth=linewidth, color=(0.5, 0.5, 0.5))
ax.plot(A, f2, linewidth=linewidth, color=colors["wave"])
ax.plot(A, f3, "--", linewidth=linewidth, color="k")

pol = [-1, 1, 0, 0, -(KD**hill_coefficient), 0]
roots = np.roots(pol)
fp_color = np.array([108, 138, 58]) / 255
ax.plot(
    0,
    0,
    "o",
    markersize=markersize,
    markeredgecolor=fp_color,
    markeredgewidth=markeredgewidth,
    markerfacecolor=np.concatenate((fp_color, [0.5])),
)
ax.plot(
    roots[1],
    roots[1],
    "o",
    markersize=markersize,
    markeredgecolor=fp_color,
    markeredgewidth=markeredgewidth,
    markerfacecolor="none",
)
ax.plot(
    roots[0],
    roots[0],
    "o",
    markersize=markersize,
    markeredgecolor=fp_color,
    markeredgewidth=markeredgewidth,
    markerfacecolor=np.concatenate((fp_color, [0.5])),
)

ax.set_xlabel("$u$ (a.u.)", fontsize=fontsize)
ax.set_ylabel("$f(u)$ (a.u.)", fontsize=fontsize)
ax.set_title("fixed point analysis\n", fontsize=fontsize)
ax = style_axes(ax, fontsize=fontsize)

In [25]:
KD = 0.4  
t_max = 50
L = 100.0
n = 2000
hill_coefficient = 4
gamma = 1.0
D = 1.0
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
ts = np.linspace(0, t_max, 10*t_max)


"""below thresh"""
B0 = 1.01 * Bc_cell_theory(hill_coefficient, KD, gamma)
model = HillModel(
    D=0, KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, gamma=gamma
)
state = State4(
    A=iw.ScalarField((1,), lb, 1, 0),
    B=iw.ScalarField((1,), lb, 1, fn=lambda x: B0 * jnp.array(x == 0).astype("float")),
)

solution = iw.solve(
    model,
    state,
    t0=0,
    t1=t_max,
    saveat=SaveAt(ts=ts),
    **kwargs,
)

A = solution.ys.A.values[:, 0]

ax = axs[0, 1]
ax.plot(ts, A, "-", linewidth=linewidth, color=(0.5, 0.5, 0.5), label="$I=0.00385$")

ax.set_xlabel("time (a.u.)", fontsize=fontsize)
ax.set_ylabel("$u$ (a.u.)", fontsize=fontsize)
ax.set_title(f"below-threshold response \n$I={B0:.3f}$", fontsize=fontsize)
ax.set_ylim([0, 1.2])
ax = style_axes(ax, fontsize=fontsize)


"""above thresh"""
B0 = 1.03 * Bc_cell_theory(hill_coefficient, KD, gamma)
model = HillModel(
    D=0, KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, gamma=gamma
)
state = State4(
    A=iw.ScalarField((1,), lb, 1, 0),
    B=iw.ScalarField((1,), lb, 1, fn=lambda x: B0 * jnp.array(x == 0).astype("float")),
)

solution = iw.solve(
    model,
    state,
    t0=0,
    t1=t_max,
    saveat=SaveAt(ts=ts),
    **kwargs,
)

A = solution.ys.A.values[:, 0]

ax = axs[0, 2]
ax.plot(ts, A, "-", linewidth=linewidth, color=colors["wave"], label="$I=0.00387$")
ax.set_xlabel("time (a.u.)", fontsize=fontsize)
ax.set_ylabel("$u$ (a.u.)", fontsize=fontsize)
ax.set_title(f"above-threshold activation \n$I={B0:.3f}$", fontsize=fontsize)
ax.set_ylim([0, 1.2])
ax = style_axes(ax, fontsize=fontsize)

### comparison with numerics --- KD

In [26]:
KD = 0.01
hill_coefficient = 4

In [27]:
"""plot"""

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/KD_sweep_cell.pkl",
    "rb",
) as f:
    KD_sweep_dict = pickle.load(f)
KDs = KD_sweep_dict["KDs"]
Bc_cell = KD_sweep_dict["Bc_cell"]

ax = axs[1, 0]

cell_theory = (
    (1 - (1 / hill_coefficient))
    * (1 / hill_coefficient) ** (1 / (hill_coefficient - 1))
    * KDs ** (hill_coefficient / (hill_coefficient - 1))
)
ax.plot(KDs, cell_theory, "-", linewidth=linewidth, color=colors["wave"], label="analytic")

ax.plot(
    KDs,
    Bc_cell,
    marker="o",
    markersize=markersize,
    markerfacecolor="none",
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=sim_color,
    label="numerics",
)


ax.legend(fontsize=fontsize)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{mixed}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel(
    "concentration scale \nof positive feedback, $K_D$ (a.u.)", fontsize=fontsize
)
ax.set_ylim([5e-4, 5e-1])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<Axes: xlabel='concentration scale \nof positive feedback, $K_D$ (a.u.)', ylabel='$I^{mixed}_{c}$ (a.u.)'>

### comparison with numerics--- vary hill coeff

In [28]:
"""plot"""

ns = np.linspace(2, 5, 100)

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/hill_coefficient_sweep.pkl",
    "rb",
) as f:
    hill_coeff_sweep_dict = pickle.load(f)
hill_coefficients = hill_coeff_sweep_dict["hill_coefficients"]
Bc_cell = hill_coeff_sweep_dict["Bc_cell"]

ax = axs[1, 1]
ax.plot(
    hill_coefficients,
    Bc_cell,
    marker="o",
    markersize=markersize,
    markerfacecolor="none",
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=sim_color,
    label="cell numerics",
)

cell_theory = (1 - (1 / ns)) * (1 / ns) ** (1 / (ns - 1)) * KD ** (ns / (ns - 1))
ax.plot(ns, cell_theory, "-", linewidth=linewidth, color=colors["wave"], label="cell theory")

ax.set_xscale("linear")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{mixed}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel("hill coefficient, $n$", fontsize=fontsize)
ax.set_ylim([1e-5, 5e-3])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<Axes: xlabel='hill coefficient, $n$', ylabel='$I^{mixed}_{c}$ (a.u.)'>

### comparision with numerics---vary gamma

In [29]:
"""plot"""

hill_coefficient = 3

# load the data
with open(
    r"/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/gamma_sweep.pkl",
    "rb",
) as f:
    gamma_sweep_dict = pickle.load(f)
gammas = gamma_sweep_dict["gammas"]
Bc_cell = gamma_sweep_dict["Bc_cell"]
gamma_theory = np.linspace(np.min(gammas), np.max(gammas), 1000)

ax = axs[1, 2]
ax.plot(
    gammas,
    Bc_cell,
    marker="o",
    markersize=markersize,
    markerfacecolor="none",
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=sim_color,
    label="cell numerics",
)

cell_theory = (
    (1 - (1 / hill_coefficient))
    * (1 / hill_coefficient) ** (1 / (hill_coefficient - 1))
    * (KD * gamma_theory) ** (hill_coefficient / (hill_coefficient - 1))
)
ax.plot(
    gamma_theory,
    cell_theory,
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="cell theory",
)

ax.set_xscale("linear")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{mixed}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel(r"decay rate, $\gamma$ (1/min)", fontsize=fontsize)
ax.set_ylim([5e-7, 1e-4])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<Axes: xlabel='decay rate, $\\gamma$ (1/min)', ylabel='$I^{mixed}_{c}$ (a.u.)'>

In [30]:
plt.gcf().tight_layout()

In [31]:
plt.savefig(
    r"/home/brandon/Documents/Code/immunowave/plots/2025-07-25_numerics_comparision_fig-mixed.pdf"
)